In [1]:
from imports import *

In [ ]:
fraction = 3
fmin=0.002
fmax=1.025
k=np.arange(np.log2(f_min), np.log2(f_max), 1/fraction)
freq_centers = 2**k

def octave_average(d,f,fmin=.002,fmax=1.025,fraction=3):
    # 1/3 octave (fraction=3) → Standard in engineering seismology and ground motion studies.
    # 1/6 octave (fraction=6) → Provides better resolution while still smoothing noise.
    # 1/12 octave (fraction=12) → High resolution, often used for detailed spectral analysis.
    # 1/8 octave (fraction=8) → Sometimes used in geophysical applications for a balance between smoothing and resolution.
    k=np.arange(np.log2(fmin), np.log2(fmax), 1/fraction)
    freq_centers = 2**k
    out=[]
    for i, fc in enumerate(freq_centers):
        f_low = fc / 2**(1/(2*fraction))
        f_high = fc * 2**(1/(2*fraction))
        # Find indices within this band
        indices = np.where((f >= f_low) & (f <= f_high))[0]
        if len(indices) > 0:out.append(np.mean(d[indices]))
    return out

In [85]:
np.diff(f)

array([0.002, 0.002, 0.002, ..., 0.002, 0.002, 0.002])

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

def fractional_octave_averaging(time_series, fs, fraction=3):
    """
    Perform fractional octave averaging on a given time series.
    
    Parameters:
        time_series (array): The input time series data.
        fs (float): Sampling frequency of the time series.
        fraction (int): Defines the fractional octave (e.g., 3 for third-octave bands).
    
    Returns:
        freq_centers (array): Center frequencies of the fractional octave bands.
        smoothed_psd (array): Smoothed power spectral density values.
    """
    
    # Compute Power Spectral Density (PSD) using Welch’s method
    f, psd = signal.welch(time_series, fs, nperseg=1024)

    # Define fractional octave band edges
    f_min, f_max = f[1], f[-1]  # Avoid DC component at f[0] = 0
    k = np.arange(np.log2(f_min), np.log2(f_max), 1/fraction)
    freq_centers = 2**k

    smoothed_psd = np.zeros_like(freq_centers)

    # Perform logarithmic averaging
    for i, fc in enumerate(freq_centers):
        f_low = fc / 2**(1/(2*fraction))
        f_high = fc * 2**(1/(2*fraction))

        # Find indices within this band
        indices = np.where((f >= f_low) & (f <= f_high))[0]

        if len(indices) > 0:smoothed_psd[i] = np.mean(psd[indices])  # Averaging in log-space

    return freq_centers, smoothed_psd

# Example usage
if __name__ == "__main__":
    # Generate synthetic time series (example: Gaussian white noise)
    fs = 1000  # Sampling frequency in Hz
    t = np.linspace(0, 10, fs*10)  # 10 seconds of data
    time_series = np.random.randn(len(t))

    # Perform fractional octave averaging (third-octave smoothing)
    freq_centers, smoothed_psd = fractional_octave_averaging(time_series, fs, fraction=3)

    # Plot the results
    plt.figure(figsize=(8,5))
    plt.semilogx(freq_centers, 10*np.log10(smoothed_psd), '-o', label="Fractional Octave Averaged PSD")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Power/Frequency (dB/Hz)")
    plt.title("Fractional Octave Averaging of Power Spectral Density")
    plt.legend()
    plt.grid(True, which="both", linestyle="--")
    plt.show()
